In [ ]:
import sys
sys.path.insert(0, '../../')

# Pipeline 4 — Dezem Anomaly Detection

**Flow:**
```
raw signal
  ↓ compute_window_features_6d  (window_length=8, window_shift=2)
features  (N, 6)  —  avg, outliers_up, outliers_down, std, alternation, slope
  ↓ ForecastingTransformer(input_dim=6)  [trained below, 48-window context]
     causal encoder → bottleneck → 6D learned representation  (N−48, 6)
  ↓ BGMM trains on training-split predictions, scores all predictions
anomaly score  (N−48,)
```
The transformer bakes 48 windows of temporal context into each predicted vector.
BGMM learns what normal predicted states look like; low probability = anomaly.

## Parameters

In [ ]:
import torch
import numpy as np

torch.manual_seed(42)
np.random.seed(42)

CSV_PATH      = "unknown"
WINDOW_LENGTH = 8      # must match BGMM config
WINDOW_SHIFT  = 2      # must match BGMM config
SEQ_LEN       = 48     # transformer history length (# of 6D windows)
TRAIN_SPLIT   = 0.8
BATCH_SIZE    = 32
NUM_EPOCHS    = 100
LR            = 1e-3
CHECKPOINT    = '../../checkpoints/forecasting_dezem_6d.pt'
DEVICE        = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SENSITIVITY   = 0.5    # BGMM sensitivity: 0=nothing flagged, 1=everything flagged
print('Device:', DEVICE)

## 1 — Load data & compute 6D features

In [ ]:
import numpy as np
from src.data.dezem import load_dezem_csv, compute_window_features_6d

_, _, meta = load_dezem_csv(CSV_PATH)
values_raw = meta['values_raw']
index      = meta['index']

feats = compute_window_features_6d(values_raw, WINDOW_LENGTH, WINDOW_SHIFT)

print(f'Signal : {len(values_raw):,} hourly points  ({index[0].date()} → {index[-1].date()})')
print(f'Features: {feats.shape}  [avg, outliers_up, outliers_down, std, alternation, slope]')

## 2b — Normalize features (fit on training split only)

In [ ]:
from sklearn.preprocessing import StandardScaler

split        = int(len(feats) * TRAIN_SPLIT)
scaler       = StandardScaler().fit(feats[:split])
feats_scaled = scaler.transform(feats).astype('float32')

print(f'Scaler fitted on {split} training windows')
print(f'Feature means  (train): {scaler.mean_.round(3)}')
print(f'Feature stds   (train): {scaler.scale_.round(3)}')

## 2 — Dataset & DataLoader

In [ ]:
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

class ForecastingDataset(Dataset):
    def __init__(self, feats: np.ndarray, seq_len: int, jitter_std: float = 0.0):
        self.feats      = torch.tensor(feats, dtype=torch.float32)
        self.seq_len    = seq_len
        self.jitter_std = jitter_std

    def __len__(self):
        return len(self.feats) - self.seq_len

    def __getitem__(self, idx):
        x = self.feats[idx : idx + self.seq_len]
        y = self.feats[idx + self.seq_len]
        if self.jitter_std > 0:
            x = x + torch.randn_like(x) * self.jitter_std
        return x, y


train_dl = DataLoader(ForecastingDataset(feats_scaled[:split], SEQ_LEN, jitter_std=0.01),
                      batch_size=BATCH_SIZE, shuffle=True)
val_dl   = DataLoader(ForecastingDataset(feats_scaled[split:], SEQ_LEN),
                      batch_size=BATCH_SIZE, shuffle=False)

print(f'Train samples : {len(train_dl.dataset)}')
print(f'Val samples   : {len(val_dl.dataset)}')

## 3 — Train ForecastingTransformer (6D)

In [ ]:
import torch.optim as optim
import matplotlib.pyplot as plt
from IPython.display import clear_output
from src.models.forecasting import ForecastingTransformer

model     = ForecastingTransformer(input_dim=6).to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
loss_fn   = nn.MSELoss()

train_losses, val_losses = [], []
best_val_loss     = float('inf')
patience          = 10
epochs_no_improve = 0

for epoch in range(NUM_EPOCHS):
    # --- train ---
    model.train()
    total_train = 0.0
    for x, y in train_dl:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        loss = loss_fn(model(x), y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_train += loss.item()

    # --- val ---
    model.eval()
    total_val = 0.0
    with torch.no_grad():
        for x, y in val_dl:
            total_val += loss_fn(model(x.to(DEVICE)), y.to(DEVICE)).item()

    scheduler.step()

    avg_train = total_train / len(train_dl)
    avg_val   = total_val   / len(val_dl)
    train_losses.append(avg_train)
    val_losses.append(avg_val)

    # --- early stopping ---
    if avg_val < best_val_loss:
        best_val_loss = avg_val
        torch.save(model.state_dict(), CHECKPOINT)
        epochs_no_improve = 0
        tag = '✓'
    else:
        epochs_no_improve += 1
        tag = f'({epochs_no_improve}/{patience})'

    # --- live plot ---
    clear_output(wait=True)
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(train_losses, label='Train', color='steelblue')
    ax.plot(val_losses,   label='Val',   color='darkorange')
    ax.axvline(len(train_losses) - 1, color='gray', linewidth=0.5, linestyle='--')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE Loss')
    ax.set_title(f'Epoch {epoch+1}/{NUM_EPOCHS} | Train {avg_train:.4f} | Val {avg_val:.4f} | LR {scheduler.get_last_lr()[0]:.2e} {tag}')
    ax.legend()
    ax.grid(True)
    plt.tight_layout()
    plt.show()

    if epochs_no_improve >= patience:
        print(f'Early stop at epoch {epoch+1}. Best val loss: {best_val_loss:.4f}')
        break

print(f'Best checkpoint saved → {CHECKPOINT}  (val loss {best_val_loss:.4f})')

## 4 — Pipeline 4: transformer predictions → BGMM

BGMM trains on the transformer's context-aware predicted vectors (training split),
then scores every prediction. Low probability under the learnt mixture = anomaly.

In [ ]:
from src.integration import run_pipeline_4

# reload from checkpoint so this cell is re-runnable independently
model_inf = ForecastingTransformer(input_dim=6)
model_inf.load_state_dict(torch.load(CHECKPOINT, map_location='cpu'))

r4 = run_pipeline_4(
    feats_scaled,          # pre-scaled — scaler already fitted above
    model_inf,
    seq_len=SEQ_LEN,
    device=DEVICE,
    train_split=TRAIN_SPLIT,
)

print(f'Predictions shape : {r4.predictions.shape}')
print(f'BGMM scores shape : {r4.bgmm_scores.shape}')
print(f'BGMM clusters     : {r4.gmm.n_components}')
print(f'Mean score        : {r4.bgmm_scores.mean():.4f}')
print(f'Max score         : {r4.bgmm_scores.max():.4f}')

## 5 — BGMM cluster means

What normal predicted states did BGMM learn?

In [ ]:
import pandas as pd

FEATURE_NAMES = ['avg', 'outliers_up', 'outliers_down', 'std', 'alternation', 'slope']

df_clusters = pd.DataFrame(r4.gmm.means[0], columns=FEATURE_NAMES)
df_clusters.index.name = 'cluster'
print(f'BGMM found {r4.gmm.n_components} clusters in transformer predictions\n')
display(df_clusters.round(3))

## 6 — Anomaly score over time

In [ ]:
from bgmm import data_preprocessing

df_raw = pd.DataFrame({'sensor': values_raw}, index=index)
ts_coll = data_preprocessing.TimeSeriesCollection.from_df(
    df_raw,
    [{'window_length': WINDOW_LENGTH, 'window_shift': WINDOW_SHIFT}],
    is_training_data=True,
    strip_unhandable_data_for_moving_window_=True,
)[0]

datapoint_indices = ts_coll.segment_idx_to_datapoint(np.arange(ts_coll.n_windows))
p4_ts = index[datapoint_indices[SEQ_LEN:]]

In [ ]:
threshold = 1 - SENSITIVITY
anomaly_indices = r4.gmm.getPointAnomalies(r4.predictions[np.newaxis], SENSITIVITY)
anomalies = np.zeros(len(r4.bgmm_scores), dtype=bool)
anomalies[anomaly_indices] = True

plt.figure(figsize=(18, 4))
plt.plot(p4_ts, r4.bgmm_scores, color='purple', linewidth=0.5, alpha=0.7)
plt.axhline(threshold, color='black', linestyle='--', linewidth=1,
            label=f'threshold (1 − sensitivity = {threshold:.2f})')
plt.scatter(p4_ts[anomalies], r4.bgmm_scores[anomalies],
            color='black', s=10, zorder=5,
            label=f'anomalies ({anomalies.sum()})')
plt.title('Pipeline 4 — Anomaly Score (BGMM on transformer predictions)')
plt.xlabel('Time')
plt.ylabel('BGMM anomaly score (0→1)')
plt.legend()
plt.tight_layout()
plt.show()
print(f'Anomalies flagged: {anomalies.sum()}')

## 7 — Anomaly report (BGMM grouped analysis)

In [ ]:
ENGLISH_MESSAGES = [
    ["avg too high",                    "avg too low"],
    ["too many upward outliers",        "too few upward outliers"],
    ["too few downward outliers",       "outliers too deep"],
    ["values spread too wide",          "values too close together"],
    ["fewer/smaller jumps than normal", "more/larger jumps than normal"],
    ["slope rising too steeply",        "slope falling too steeply"],
]

anom_info = r4.gmm.getAnomalieInfo(
    r4.predictions[np.newaxis],
    FEATURE_NAMES,
    SENSITIVITY,
    dist_to_next_anomaly=50,
)

groups = anom_info[0]
n_groups = sum(len(v) for v in groups.values())
print(f"Anomaly groups found: {n_groups}\n")
print(f"{'Start':<22} {'End':<22} {'Points':>6}  Top driving features")
print("-" * 90)

for start_idx, group_list in sorted(groups.items()):
    for group in group_list:
        start_ts = p4_ts[group['start']]
        end_ts   = p4_ts[group['end']]

        top_feats = sorted(group['sum_anom_vec'].items(), key=lambda x: abs(x[1]), reverse=True)[:2]
        feat_desc = []
        for feat_name, val in top_feats:
            feat_idx  = FEATURE_NAMES.index(feat_name)
            direction = 0 if val < 0 else 1
            feat_desc.append(f"{feat_name} ({ENGLISH_MESSAGES[feat_idx][direction]})")

        print(f"{str(start_ts):<22} {str(end_ts):<22} {group['n_anomalies']:>6}  {' | '.join(feat_desc)}")

## 8 — Baseline P1: BGMM directly on raw 6D features

No transformer. BGMM sees the raw window statistics directly.
Comparing P1 vs P4 shows what the transformer's learned representation adds.

In [ ]:
from bgmm import getModel as bgmm_getModel

# P1: BGMM on raw (but scaled) 6D features — same scaler as P4 for fair comparison
n_train_p1   = int(len(feats_scaled) * TRAIN_SPLIT)
train_feats  = feats_scaled[:n_train_p1][np.newaxis]  # (1, n_train, 6)

gmm_p1 = bgmm_getModel(
    train_feats,
    start_n_components=20,
    class_portion_threshold=0.003,
    weight_concentration_prior=1e-5,
)

p1_scores = gmm_p1.anomalityScore(feats_scaled[np.newaxis])  # (N,)
p1_ts     = index[datapoint_indices]                          # full feature timeline

print(f'P1 clusters  : {gmm_p1.n_components}')
print(f'P1 mean score: {p1_scores.mean():.4f}')
print(f'P1 max score : {p1_scores.max():.4f}')

## 9 — P1 vs P4 comparison plot

In [ ]:
threshold = 1 - SENSITIVITY

fig, axes = plt.subplots(3, 1, figsize=(18, 10), sharex=True)

# --- raw signal ---
axes[0].plot(index, values_raw, color='steelblue', linewidth=0.4, alpha=0.8)
axes[0].set_ylabel('Energy (kWh)')
axes[0].set_title('Raw signal')
axes[0].grid(True, alpha=0.3)

# --- P1 ---
p1_anom_idx = gmm_p1.getPointAnomalies(feats_scaled[np.newaxis], SENSITIVITY)
p1_anom     = np.zeros(len(p1_scores), dtype=bool)
p1_anom[p1_anom_idx] = True

axes[1].plot(p1_ts, p1_scores, color='steelblue', linewidth=0.5, alpha=0.7)
axes[1].axhline(threshold, color='black', linestyle='--', linewidth=0.8)
axes[1].scatter(p1_ts[p1_anom], p1_scores[p1_anom], color='red', s=8, zorder=5,
                label=f'P1 anomalies ({p1_anom.sum()})')
axes[1].set_ylabel('BGMM score')
axes[1].set_title(f'P1 — BGMM on raw 6D features  |  clusters: {gmm_p1.n_components}  |  anomalies: {p1_anom.sum()}')
axes[1].set_ylim(-0.05, 1.05)
axes[1].legend(loc='upper right')
axes[1].grid(True, alpha=0.3)

# --- P4 ---
p4_anom_idx = r4.gmm.getPointAnomalies(r4.predictions[np.newaxis], SENSITIVITY)
p4_anom     = np.zeros(len(r4.bgmm_scores), dtype=bool)
p4_anom[p4_anom_idx] = True

axes[2].plot(p4_ts, r4.bgmm_scores, color='purple', linewidth=0.5, alpha=0.7)
axes[2].axhline(threshold, color='black', linestyle='--', linewidth=0.8)
axes[2].scatter(p4_ts[p4_anom], r4.bgmm_scores[p4_anom], color='red', s=8, zorder=5,
                label=f'P4 anomalies ({p4_anom.sum()})')
axes[2].set_ylabel('BGMM score')
axes[2].set_title(f'P4 — BGMM on transformer predictions  |  clusters: {r4.gmm.n_components}  |  anomalies: {p4_anom.sum()}')
axes[2].set_ylim(-0.05, 1.05)
axes[2].legend(loc='upper right')
axes[2].grid(True, alpha=0.3)

plt.xlabel('Time')
plt.tight_layout()
plt.show()

# --- correlation ---
# align P1 and P4 to the same time window (P4 starts seq_len steps later)
p1_aligned = p1_scores[SEQ_LEN:]
p4_aligned = r4.bgmm_scores
corr = np.corrcoef(p1_aligned, p4_aligned)[0, 1]
print(f'Pearson correlation between P1 and P4 scores (aligned): {corr:.4f}')
print(f'  P1 flagged {p1_anom.sum()} anomalies over {len(p1_scores)} windows')
print(f'  P4 flagged {p4_anom.sum()} anomalies over {len(r4.bgmm_scores)} windows')